# Pol-ISARSpaceTarget OBB Detection: Learnable Polarimetric Fusion

This notebook extends the previous object detection experiment.

Comparisons:

1. `single_vv`: YOLO-OBB on VV amplitude image.
2. `pauli_fusion`: YOLO-OBB on fixed full-pol Pauli fusion.
3. `learned_fusion`: YOLO-OBB on images generated by a small learnable
   polarimetric fusion stem.

Learnable fusion idea:

- Input: HH, HV, VH, VV complex channels represented as real/imag channels.
- A small CNN fusion stem maps 8 channels to 3 RGB-like channels.
- It is trained using component OBB labels converted to dense supervision:
  foreground mask, boundary mask, and scattering-energy target.
- The learned 3-channel images are then exported and used as YOLO-OBB input.


In [ ]:
!pip -q install ultralytics pillow pyyaml pandas matplotlib opencv-python scikit-image


In [ ]:
import json
import math
import os
import re
import shutil
import zipfile
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from ultralytics import YOLO

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


## Paths


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", exc)

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0"),
    Path("/content/Pol_ISARSpaceTarget1.0"),
    Path("C:/Users/keerthish/Desktop/drdo/Pol_ISARSpaceTarget1.0"),
]
DATASET_ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), ROOT_CANDIDATES[0])
print("DATASET_ROOT:", DATASET_ROOT)

LABELTXT_DIR = DATASET_ROOT / "ImageSets" / "labelTxt"
DAT_DIR = DATASET_ROOT / "ImageSets" / "images"
TIFF_DIR = DATASET_ROOT / "TIFFImages" / "AmplitudeImages"

assert LABELTXT_DIR.exists(), f"Missing labels: {LABELTXT_DIR}"
assert DAT_DIR.exists(), f"Missing complex dat files: {DAT_DIR}"

OUT_ROOT = Path("/content/polisar_obb_yolo")
DRIVE_OUT = Path("/content/drive/MyDrive/drdo/polisar_obb_outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)


## Label Reader and Dataset Summary


In [ ]:
def read_dota_label_file(path: Path) -> List[Tuple[List[float], str, int]]:
    rows = []
    if not path.exists() or not path.read_text().strip():
        return rows
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 10:
            continue
        coords = [float(x) for x in parts[:8]]
        cls_name = parts[8]
        difficult = int(float(parts[9]))
        rows.append((coords, cls_name, difficult))
    return rows


class_names = sorted(
    {
        cls
        for split in ["train", "valid", "test"]
        for txt in (LABELTXT_DIR / split).glob("*.txt")
        for _, cls, _ in read_dota_label_file(txt)
    }
)
class_to_id = {name: idx for idx, name in enumerate(class_names)}
print("Class map:", class_to_id)

split_summary = []
for split in ["train", "valid", "test"]:
    txt_files = sorted((LABELTXT_DIR / split).glob("*.txt"))
    obj_count = sum(len(read_dota_label_file(p)) for p in txt_files)
    split_summary.append({"split": split, "images": len(txt_files), "objects": obj_count})
display(pd.DataFrame(split_summary))

(DRIVE_OUT / "obb_class_names.json").write_text(json.dumps(class_to_id, indent=2))


## Complex Data Readers and Baseline Fusion Builders


In [ ]:
POL_SUFFIXES = ["HH", "HV", "VH", "VV"]


def robust_uint8(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = np.log1p(np.maximum(x, 0.0))
    lo, hi = np.percentile(x, [1, 99])
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    return (255 * x).astype(np.uint8)


def base_name_candidates(base_name: str) -> List[str]:
    cleaned = re.sub(r"\s+\(\d+\)$", "", base_name)
    return [base_name, cleaned] if cleaned != base_name else [base_name]


def find_first_existing(paths: Sequence[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def read_complex_dat(path: Path, shape: Tuple[int, int] = (256, 256)) -> np.ndarray:
    raw = np.fromfile(path, dtype=np.float32)
    real = raw[0::2].reshape(shape)
    imag = raw[1::2].reshape(shape)
    return real + 1j * imag


def find_dat(base_name: str, pol: str, split: str) -> Path:
    path = find_first_existing([DAT_DIR / split / f"{candidate}_{pol}.dat" for candidate in base_name_candidates(base_name)])
    if path is None:
        raise FileNotFoundError(f"No {pol}.dat found for {base_name} in split {split}")
    return path


def load_full_pol_complex(base_name: str, split: str) -> Dict[str, np.ndarray]:
    return {pol: read_complex_dat(find_dat(base_name, pol, split)) for pol in POL_SUFFIXES}


def build_single_vv(base_name: str, split: str) -> np.ndarray:
    tiff_path = find_first_existing([TIFF_DIR / f"{candidate}_VV.tiff" for candidate in base_name_candidates(base_name)])
    if tiff_path is not None:
        img = np.array(Image.open(tiff_path).convert("L"))
        return np.stack([img, img, img], axis=-1)
    vv = read_complex_dat(find_dat(base_name, "VV", split))
    img = robust_uint8(np.abs(vv))
    return np.stack([img, img, img], axis=-1)


def build_pauli_fusion(base_name: str, split: str) -> np.ndarray:
    c = load_full_pol_complex(base_name, split)
    hh, hv, vh, vv = c["HH"], c["HV"], c["VH"], c["VV"]
    r = robust_uint8(np.abs(hh - vv))
    g = robust_uint8(np.abs(hv + vh))
    b = robust_uint8(np.abs(hh + vv))
    return np.stack([r, g, b], axis=-1)


def full_pol_to_8ch(base_name: str, split: str) -> np.ndarray:
    c = load_full_pol_complex(base_name, split)
    chans = []
    for pol in POL_SUFFIXES:
        arr = c[pol]
        chans.append(arr.real.astype(np.float32))
        chans.append(arr.imag.astype(np.float32))
    x = np.stack(chans, axis=0)
    # Per-image robust scaling for fusion-stem input only.
    med = np.median(x, axis=(1, 2), keepdims=True)
    q1 = np.percentile(x, 25, axis=(1, 2), keepdims=True)
    q3 = np.percentile(x, 75, axis=(1, 2), keepdims=True)
    x = np.clip((x - med) / (q3 - q1 + 1e-6), -5, 5) / 5.0
    return x.astype(np.float32)


## Dense Supervision for Learnable Fusion

OBB labels are converted into:

- foreground mask: filled oriented boxes.
- boundary mask: object outlines.
- scattering-energy target: normalized total full-pol magnitude.


In [ ]:
def obb_masks(label_path: Path, size: int = 256) -> Tuple[np.ndarray, np.ndarray]:
    fg = np.zeros((size, size), dtype=np.float32)
    edge = np.zeros((size, size), dtype=np.float32)
    for coords, cls_name, difficult in read_dota_label_file(label_path):
        if difficult == 1:
            continue
        pts = np.array(coords, dtype=np.float32).reshape(4, 2)
        pts = np.round(pts).astype(np.int32)
        cv2.fillPoly(fg, [pts], 1.0)
        cv2.polylines(edge, [pts], True, 1.0, thickness=2)
    return fg, edge


def energy_target_from_8ch(x: np.ndarray) -> np.ndarray:
    real = x[0::2]
    imag = x[1::2]
    mag = np.sqrt(np.sum(real * real + imag * imag, axis=0))
    mag = np.log1p(np.maximum(mag, 0))
    lo, hi = np.percentile(mag, [1, 99])
    mag = np.clip((mag - lo) / (hi - lo + 1e-6), 0, 1)
    return mag.astype(np.float32)


class FusionStemDataset(Dataset):
    def __init__(self, split: str, max_samples: int | None = None):
        self.split = split
        self.label_paths = sorted((LABELTXT_DIR / split).glob("*.txt"))
        if max_samples is not None:
            self.label_paths = self.label_paths[:max_samples]

    def __len__(self) -> int:
        return len(self.label_paths)

    def __getitem__(self, idx: int):
        label_path = self.label_paths[idx]
        base = label_path.stem
        x = full_pol_to_8ch(base, self.split)
        fg, edge = obb_masks(label_path)
        energy = energy_target_from_8ch(x)
        target = np.stack([fg, edge, energy], axis=0).astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(target), base


class PolarimetricFusionStem(nn.Module):
    def __init__(self, in_channels: int = 8, hidden: int = 24, out_channels: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, out_channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


## Train Learnable Fusion Stem

This is not the final detector. It learns a task-informed 3-channel
polarimetric representation that is then used to train YOLO-OBB.


In [ ]:
FUSION_EPOCHS = 8
FUSION_BATCH = 16
FUSION_MAX_TRAIN = None

fusion_model = PolarimetricFusionStem().to(DEVICE)
train_fusion_ds = FusionStemDataset("train", max_samples=FUSION_MAX_TRAIN)
valid_fusion_ds = FusionStemDataset("valid", max_samples=None)
train_fusion_loader = DataLoader(train_fusion_ds, batch_size=FUSION_BATCH, shuffle=True, num_workers=0)
valid_fusion_loader = DataLoader(valid_fusion_ds, batch_size=FUSION_BATCH, shuffle=False, num_workers=0)

opt = torch.optim.AdamW(fusion_model.parameters(), lr=1e-3, weight_decay=1e-4)
bce = nn.BCELoss()
fusion_history = []

for epoch in range(FUSION_EPOCHS):
    fusion_model.train()
    train_loss = 0.0
    for xb, yb, _ in train_fusion_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = fusion_model(xb)
        loss = bce(pred, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        train_loss += float(loss.detach().cpu())
    train_loss /= max(len(train_fusion_loader), 1)

    fusion_model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for xb, yb, _ in valid_fusion_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            valid_loss += float(bce(fusion_model(xb), yb).detach().cpu())
    valid_loss /= max(len(valid_fusion_loader), 1)
    fusion_history.append({"epoch": epoch + 1, "train_loss": train_loss, "valid_loss": valid_loss})
    print(f"fusion epoch {epoch + 1}/{FUSION_EPOCHS} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f}")

torch.save(fusion_model.state_dict(), DRIVE_OUT / "learnable_fusion_stem.pt")
pd.DataFrame(fusion_history).to_csv(DRIVE_OUT / "learnable_fusion_stem_history.csv", index=False)

hist = pd.DataFrame(fusion_history)
plt.figure(figsize=(6, 4))
plt.plot(hist["epoch"], hist["train_loss"], marker="o", label="train")
plt.plot(hist["epoch"], hist["valid_loss"], marker="o", label="valid")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Learnable Fusion Stem Training")
plt.legend()
plt.tight_layout()
plt.savefig(DRIVE_OUT / "learnable_fusion_stem_loss.png", dpi=160)
plt.show()


## Convert Labels to YOLO-OBB Format


In [ ]:
def convert_label_to_yolo_obb(src_txt: Path, dst_txt: Path, width: int = 256, height: int = 256) -> int:
    out_lines = []
    for coords, cls_name, difficult in read_dota_label_file(src_txt):
        if difficult == 1:
            continue
        cls_id = class_to_id[cls_name]
        norm = []
        for idx, value in enumerate(coords):
            norm.append(value / width if idx % 2 == 0 else value / height)
        norm = [min(max(v, 0.0), 1.0) for v in norm]
        out_lines.append(" ".join([str(cls_id)] + [f"{v:.6f}" for v in norm]))
    dst_txt.parent.mkdir(parents=True, exist_ok=True)
    dst_txt.write_text("\n".join(out_lines) + ("\n" if out_lines else ""))
    return len(out_lines)


def write_data_yaml(out_dir: Path) -> Path:
    data_yaml = {
        "path": str(out_dir),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": {idx: name for idx, name in enumerate(class_names)},
    }
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
    return yaml_path


def build_yolo_dataset(variant: str, max_per_split: int | None = None) -> Path:
    out_dir = OUT_ROOT / variant
    if out_dir.exists():
        shutil.rmtree(out_dir)
    for split in ["train", "valid", "test"]:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        txt_files = sorted((LABELTXT_DIR / split).glob("*.txt"))
        if max_per_split is not None:
            txt_files = txt_files[:max_per_split]

        for i, txt in enumerate(txt_files, start=1):
            base = txt.stem
            if variant == "single_vv":
                img = build_single_vv(base, split)
            elif variant == "pauli_fusion":
                img = build_pauli_fusion(base, split)
            elif variant == "learned_fusion":
                x = torch.from_numpy(full_pol_to_8ch(base, split)).unsqueeze(0).to(DEVICE)
                fusion_model.eval()
                with torch.no_grad():
                    pred = fusion_model(x).squeeze(0).detach().cpu().numpy()
                img = np.moveaxis((255 * np.clip(pred, 0, 1)).astype(np.uint8), 0, -1)
            else:
                raise ValueError(variant)

            cv2.imwrite(str(out_dir / "images" / split / f"{base}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            convert_label_to_yolo_obb(txt, out_dir / "labels" / split / f"{base}.txt")
            if i % 100 == 0:
                print(variant, split, i, "/", len(txt_files))

    yaml_path = write_data_yaml(out_dir)
    print("Prepared:", out_dir)
    return yaml_path


## Build Three YOLO-OBB Datasets


In [ ]:
MAX_PER_SPLIT = None
single_yaml = build_yolo_dataset("single_vv", max_per_split=MAX_PER_SPLIT)
pauli_yaml = build_yolo_dataset("pauli_fusion", max_per_split=MAX_PER_SPLIT)
learned_yaml = build_yolo_dataset("learned_fusion", max_per_split=MAX_PER_SPLIT)


## Visual Sanity Check


In [ ]:
def show_example(variant: str, split: str = "train") -> None:
    img_path = next((OUT_ROOT / variant / "images" / split).glob("*.png"))
    label_path = OUT_ROOT / variant / "labels" / split / f"{img_path.stem}.txt"
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    h, w = img.shape[:2]
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        pts = np.array([float(x) for x in parts[1:]], dtype=np.float32).reshape(4, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        pts = np.vstack([pts, pts[0]])
        plt.plot(pts[:, 0], pts[:, 1], linewidth=1.5)
    plt.title(f"{variant}: {img_path.name}")
    plt.axis("off")
    plt.show()


for variant in ["single_vv", "pauli_fusion", "learned_fusion"]:
    show_example(variant)


## Train and Test YOLO-OBB

If the single-VV and Pauli-fusion runs already exist, you may skip retraining
those and train only `learned_fusion`. For a final comparison, keep all three
under the same settings.


In [ ]:
EPOCHS = 50
IMGSZ = 256
BATCH = 16
MODEL_NAME = "yolo11n-obb.pt"


def train_obb_experiment(name: str, yaml_path: Path, model_name: str = MODEL_NAME, epochs: int = EPOCHS):
    model = YOLO(model_name)
    train_results = model.train(
        data=str(yaml_path),
        task="obb",
        epochs=epochs,
        imgsz=IMGSZ,
        batch=BATCH,
        project=str(DRIVE_OUT / "runs_obb"),
        name=name,
        exist_ok=True,
        patience=15,
        plots=True,
    )
    test_metrics = model.val(data=str(yaml_path), split="test", task="obb", imgsz=IMGSZ)
    return train_results, test_metrics


RUN_SINGLE_AND_PAULI_AGAIN = False

if RUN_SINGLE_AND_PAULI_AGAIN:
    single_results, single_metrics = train_obb_experiment("baseline_single_vv_yolo_obb_rerun", single_yaml)
    pauli_results, pauli_metrics = train_obb_experiment("fixed_pauli_fusion_yolo_obb_rerun", pauli_yaml)

learned_results, learned_metrics = train_obb_experiment("proposed_learned_fusion_yolo_obb", learned_yaml)


## Comparison Table

Fill known previous results if you did not rerun `single_vv` and
`pauli_fusion`. The learned-fusion run directory will contain `results.csv`
and validation/test outputs from Ultralytics.


In [ ]:
comparison = [
    {
        "method": "single_vv_baseline",
        "input": "VV amplitude",
        "test_precision": 0.804,
        "test_recall": 0.819,
        "test_mAP50": 0.821,
        "test_mAP50_95": 0.582,
        "note": "Previous completed run",
    },
    {
        "method": "fixed_pauli_fusion",
        "input": "R=|HH-VV|, G=|HV+VH|, B=|HH+VV|",
        "test_precision": 0.828,
        "test_recall": 0.739,
        "test_mAP50": 0.783,
        "test_mAP50_95": 0.573,
        "note": "Previous completed run",
    },
    {
        "method": "learned_fusion",
        "input": "8-channel real/imag -> learned 3-channel fusion -> YOLO-OBB",
        "test_precision": None,
        "test_recall": None,
        "test_mAP50": None,
        "test_mAP50_95": None,
        "note": "Fill from proposed_learned_fusion_yolo_obb test output after run",
    },
]
comparison_df = pd.DataFrame(comparison)
display(comparison_df)
comparison_df.to_csv(DRIVE_OUT / "obb_fusion_method_comparison_template.csv", index=False)

summary = {
    "experiment": "Pol-ISARSpaceTarget OBB detection with learnable polarimetric fusion",
    "baseline_single_vv": "YOLO-OBB on VV amplitude image",
    "baseline_pauli": "YOLO-OBB on fixed Pauli fusion image",
    "proposed_learned_fusion": "Box-supervised 8-channel real/imag polarimetric fusion stem producing 3-channel YOLO input",
    "outputs": str(DRIVE_OUT),
}
(DRIVE_OUT / "learnable_fusion_experiment_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
